# ДЗ2. Multi-object tracking на Anti-UAV: SORT, адаптация и улучшение

Используется **учебный MOT-бенчмарк на основе ИК-видео Anti-UAV**. Это не официальный протокол Multi-UAV Tracking Challenge. Исходные последовательности предоставляются в формате `val/images` и `val/labels`, где строка GT: `class_id xc yc w h track_id` (нормированные координаты). Все решения должны работать без GPU и без обучения детектора.

**Важно:** шестая колонка — track_id, использовать её можно только в `evaluate` или для визуализации GT, но **никогда в трекере**. `tracks.txt` — метаданные каталога, а не трекерные детекции.

[Ссылка на данные](https://mega.nz/file/HQJnRAzb#w9aaP5xxNqQMUX79uzpKIsYeWH6pzIowb4zz71FyPuA)

## Данные и честная оценка
- `val`: доступны GT и метрики; выбор параметров **только здесь**.
- `test`: входные детекции доступны; GT скрыт; переобучение параметров по тесту запрещено.

**Режимы входа** (единый интерфейс `detections: [x1,y1,x2,y2,score]`):
1. `ideal`: GT-боксы без ID и `score=1`;
2. `noisy`: те же GT-боксы с шумом координат, FP и FN, фиксированный seed;

**Основные метрики:** HOTA, DetA, AssA; дополнительные MOTA, IDF1, IDSW. Основная цель — исследовать ошибки ассоциации при контролируемых и реальных детекциях.

## 0. Загрузка и контроль схемы (не оценивается отдельно)
Положите `tracking_utils.py` рядом с ноутбуком. Укажите корень данных и выберите одну последовательность. В финальных экспериментах повторите расчёт по **всем** последовательностям val и приведите макроусреднение по последовательностям (дополнительно можно указать aggregate-вариант).

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tracking_utils import (discover_sequences, strip_ids, noisy_measurements,
    SortLikeTracker, run_tracker, evaluate, save_mot)
DATA_DIR=Path(#TODO)
sequences=discover_sequences(DATA_DIR)
seq_name,frames=next(iter(sequences.items()))
print(seq_name, len(frames), 'кадры',frames[0].number,'...',frames[-1].number)
assert strip_ids(frames[0].gt).shape[1]==5

4au_01_1751_0250-1750 1500 кадры 1 ... 1500


## Часть 1. Baseline (1.0 балл)
**Задача:** запустить учебный SORT-подобный baseline на ideal/noisy; получить таблицу HOTA, DetA, AssA, IDF1, MOTA, IDSW; объяснить хотя бы один конкретный сбой с номером кадра и картинкой/таблицей.

Оценивание: **0.3** корректность и отсутствие утечки ID, **0.4** метрики и воспроизводимость, **0.3** содержательный анализ конкретной ошибки.

Разрешается стартовать от `SortLikeTracker` и затем изменить его код. Для 3 баллов необходимо собственное осмысленное улучшение, а не только подбор порогов.

In [ ]:
# TODO 1: запустите baseline на ideal и noisy с одинаковыми настройками.
base_ideal = run_tracker(frames, SortLikeTracker(association='iou',iou_threshold=.1), mode='ideal')
base_noisy = run_tracker(frames, SortLikeTracker(association='iou',iou_threshold=.1), mode='noisy', seed=42)
rows=[]
for mode,pred in [('ideal',base_ideal),('noisy',base_noisy)]:
    rows.append({'mode':mode,**evaluate(frames,pred)})
pd.DataFrame(rows)[['mode','HOTA','DetA','AssA','IDF1','MOTA','IDSW']]

**TODO 1.2 — разбор сбоя:** укажите `seq_name`, номер кадра, GT и выданные треком ID; объясните, была ли причиной отсутствующая детекция, ложная детекция, gating или ошибка ассоциации. Желательно показать предыдущий/следующий кадры.

Ответ: ...

## Часть 2. Адаптация (1.0 балл)
Реализуйте/используйте **две сопоставимые функции стоимости**: IoU и расстояние центров (по желанию — Махаланобис).

Проведите контролируемые эксперименты с `max_age`, `min_hits`, а также минимум одним параметром фильтра (`meas_var` или `accel_var`). Ограничьте сетку параметров и фиксируйте seed, датасет и threshold. Выбирайте лучший вариант на val, а не на test.

Оценивание: **0.4** корректное сравнение функций стоимости, **0.4** контролируемая таблица из ≥4 конфигураций, **0.2** объяснение ошибок и вывод.

**Внимание:** числа порогов для IoU и center-distance несопоставимы напрямую, нужны самостоятельные значения.

In [ ]:
# TODO 2: замените example configs собственным обоснованным планом эксперимента.
configs = [
    {'association':'iou','iou_threshold':.1,'max_age':3,'min_hits':1,'meas_var':9.,'accel_var':2.},
    {'association':'iou','iou_threshold':.3,'max_age':3,'min_hits':1,'meas_var':9.,'accel_var':2.},
    {'association':'center','distance_threshold':20.,'max_age':3,'min_hits':1,'meas_var':9.,'accel_var':2.},
    {'association':'center','distance_threshold':35.,'max_age':5,'min_hits':2,'meas_var':16.,'accel_var':4.},
]
experiments=[]
for cfg in configs:
    pred=run_tracker(frames,SortLikeTracker(**cfg),mode='noisy',seed=42)
    score=evaluate(frames,pred)
    experiments.append({**cfg,**{k:score[k] for k in ('HOTA','DetA','AssA','IDF1','MOTA','IDSW')}})
pd.DataFrame(experiments).round(3)

**TODO 2.2 — вывод:** влияет ли выбранная конфигурация преимущественно на DetA или на AssA? Какие ошибки остаются?

## Часть 3. Собственное улучшение (1.0 балл)
Выберите **одно** направление:

- *ByteTrack-style two-pass*: на первом проходе связывайте high-confidence детекции с существующими треками, на втором — оставшиеся треки с low-confidence; **новые треки создавайте только из high-confidence**. Этот вариант имеет смысл в режиме с осмысленными confidence (например, real detector), а не при `ideal` со score=1.
- *Mahalanobis gating*: используйте инновацию `z-Hx` и ковариацию `S` фильтра, а не постоянный порог в пикселях; обоснуйте размерность и порог.
- *Продвинутая модель движения*: например, адаптивный шум или обработка разных `dt`; сравните со старым CV.
- *Greedy vs Hungarian*: удерживайте **ту же матрицу стоимости и тот же gating**, иначе эксперимент некорректен.
- Собственная обоснованная модификация после согласования с преподавателем.

Оценивание: **0.5** корректная реализация, **0.3** сравнение baseline / adaptation / improvement на одинаковых данных, **0.2** анализ выигрышей, проигрышей и ограничений.

**Запрещено:** использовать GT-ID при назначении ID; вручную исправлять идентичности по GT; подбирать параметры на test.

In [ ]:
# TODO 3: импортируйте/реализуйте улучшенный трекер.
# Пример интерфейса: tracker.step(detections_xyxy_score, frame_number) -> np.ndarray[N, 5] xyxy+track_id
# class ImprovedTracker(SortLikeTracker):
#     ...
# improved = run_tracker(frames, ImprovedTracker(...), mode='noisy', seed=42)
# print(evaluate(frames, improved))

## Часть 4. Итоговый отчёт и сдача
1. Таблица `ideal` vs `noisy` (и `detector`, если выдан) для baseline и лучшей версии.
2. Таблица контролируемой абляции: что изменили → почему → изменение HOTA/DetA/AssA.
3. Разбор ≥1 ошибки ассоциации или фрагментации с иллюстрацией.
4. Описание гиперпараметров, фиксированный seed, среднее по val-последовательностям.
5. `predictions/<sequence>.txt` в MOT-подобном формате **для каждой тестовой последовательности**.
6. Notebook с воспроизводимым запуском. Веса нейросети не нужны.

**Итоговая проверка:** 3.0 балла за выполненные части. Скрытый test — проверка обобщения, не дополнительный «неизвестный порог HOTA».

In [ ]:
# TODO 4: сохраните финальные предсказания.
# В демонстрационном варианте сохраняем baseline, замените на свою лучшую модель.
save_mot(Path('predictions') / (seq_name+'.txt'), frames, base_noisy)
print('Файл результатов подготовлен')

`tracking_utils.evaluate` — **учебная открытая реализация**. Для формальной проверки HOTA и остальных MOT-метрик в реальных задачах используйте TrackEval.